# **Stratified sample**

In [1]:
import pandas as pd

In [2]:
def criar_amostra_estratificada(df_ocorrencias: pd.DataFrame, 
                               fracao: float = 0.05, 
                               random_seed: int = 42) -> pd.DataFrame:
    """
    Cria uma amostra aleatória estratificada de um DataFrame.

    A estratificação é baseada na combinação única das colunas
    'entrevista', 'contexto_anterior' e 'contexto_posterior'.
    Isso garante que a amostra de 'fracao' (ex: 0.20 para 20%) 
    mantenha as proporções de contextos por entrevistado.

    Args:
        df_ocorrencias (pd.DataFrame): O DataFrame de entrada.
        fracao (float): A fração do DataFrame a ser amostrada (ex: 0.20).
        random_seed (int): Semente aleatória para reprodutibilidade.

    Returns:
        pd.DataFrame: Uma amostra estratificada.
    """
    
    df_ocorrencias['__entrevista_str'] = df_ocorrencias['entrevista'].astype(str)
    
    df_ocorrencias['__ctx_ant_str'] = df_ocorrencias['contexto_anterior'].fillna('NA').astype(str)
    df_ocorrencias['__ctx_pos_str'] = df_ocorrencias['contexto_posterior'].fillna('NA').astype(str)
    
    chaves_agrupamento = ['__entrevista_str', '__ctx_ant_str', '__ctx_pos_str']
    
    amostra_estratificada = df_ocorrencias.groupby(
        chaves_agrupamento, 
        group_keys=False
    ).sample(
        frac=fracao, 
        random_state=random_seed,
        replace=False 
    )

    colunas_para_remover = ['__entrevista_str', '__ctx_ant_str', '__ctx_pos_str']
    amostra_final = amostra_estratificada.drop(columns=colunas_para_remover)

    return amostra_final.sort_index()

## ***nh***

In [3]:
df_nh = pd.read_csv("../data/raw/nasal-palatal_occurrences.csv")

amostra_nh = criar_amostra_estratificada(df_nh, fracao=0.03)
print(f"Tamanho original: {len(df_nh)}")
print(f"Tamanho da amostra: {len(amostra_nh)}")

Tamanho original: 8004
Tamanho da amostra: 157


In [4]:
amostra_nh.to_csv("../data/processed/nasal-palatal_stratified_sample.csv", index=False)